# 🦀 Daadima TTS Pre-generation

Generates grandmotherly-voice Telugu audio for all 150 stories using GPU.
Output: 128kbps MP3 files uploaded to Cloudinary.

**Works on:** Google Colab (T4) / Kaggle (P100/T4)

---

## Step 1: Set up Cloudinary credentials

**Colab:** Run the cell below and enter your Cloudinary API keys when prompted.
**Kaggle:** Add secrets named `CLOUDINARY_CLOUD_NAME`, `CLOUDINARY_API_KEY`, `CLOUDINARY_API_SECRET` via Notebook → Add-ons → Secrets.

In [ ]:
import os, json, sys

# ── Detect platform ──
IN_COLAB = "google.colab" in sys.modules
IN_KAGGLE = "kaggle_secrets" in sys.modules or os.path.exists("/kaggle/")

def setup_credentials():
    if IN_KAGGLE:
        from kaggle_secrets import UserSecretsClient
        secrets = UserSecretsClient()
        for k in ["CLOUDINARY_CLOUD_NAME", "CLOUDINARY_API_KEY", "CLOUDINARY_API_SECRET"]:
            os.environ[k] = secrets.get_secret(k)
    elif IN_COLAB:
        from google.colab import userdata
        for k in ["CLOUDINARY_CLOUD_NAME", "CLOUDINARY_API_KEY", "CLOUDINARY_API_SECRET"]:
            try:
                os.environ[k] = userdata.get(k)
            except userdata.SecretNotFoundError:
                os.environ[k] = input(f"Enter {k}: ").strip()
    else:
        print("⚠ Unknown platform — set CLOUDINARY_* env vars manually")

setup_credentials()

print(f"  Cloud name: {os.environ.get('CLOUDINARY_CLOUD_NAME', '?')}")
print(f"  API key:    {os.environ.get('CLOUDINARY_API_KEY', '?')[:6]}...")
print(f"  Secret:     {'✓ set' if os.environ.get('CLOUDINARY_API_SECRET') else '✗ missing'}")

## Step 2: Install dependencies

Installs PyTorch, parler-tts, Cloudinary SDK, and other requirements.

In [ ]:
import subprocess, sys

def run(cmd):
    print(f"$ {cmd}")
    subprocess.run(cmd, shell=True, check=True)

# PyTorch (already installed on Colab/Kaggle, but ensure CUDA)
run(f"{sys.executable} -m pip install --quiet --upgrade pip")

# Install from requirements
run(f"{sys.executable} -m pip install --quiet cloudinary soundfile nltk")
run(f"{sys.executable} -m pip install --quiet git+https://github.com/huggingface/parler-tts.git")

import torch
print(f"\nPyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    print("⚠ No GPU detected! CPU will be very slow.")

## Step 3: Configure batch

Set your batch range. Split the 150 stories across platforms:

| Platform | Start | Count | Stories |
|----------|-------|-------|---------|
| Colab    | 0     | 75    | #1–#75  |
| Kaggle   | 75    | 75    | #76–#150 |

Adjust based on how much time you have (~2 min per story on GPU).

In [ ]:
# ── EDIT THESE VALUES ──
BATCH_START = 0
BATCH_COUNT = 75
LANG = "Telugu"
# ──────────────────────

print(f"Range: stories #{BATCH_START + 1}–{BATCH_START + BATCH_COUNT}")
print(f"Lang:  {LANG}")

## Step 4: Set up persistent checkpoint storage

**Colab:** Mounts Google Drive so checkpoints survive session restarts.
**Kaggle:** Saves to `/kaggle/working/` (persistent within session).

In [ ]:
CHECKPOINT_DIR = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    CHECKPOINT_DIR = "/content/drive/MyDrive/daadima-tts-checkpoints"
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
elif IN_KAGGLE:
    CHECKPOINT_DIR = "/kaggle/working"
else:
    CHECKPOINT_DIR = os.getcwd()

print(f"Checkpoints: {CHECKPOINT_DIR}")

## Step 5: Download the GPU script and story files

In [ ]:
import requests

REPO_BASE = "https://raw.githubusercontent.com/AvengerChaitu/kahaniyaan/main"

# Download GPU script
resp = requests.get(f"{REPO_BASE}/scripts/pregen_tts_gpu.py", timeout=30)
resp.raise_for_status()
with open("pregen_tts_gpu.py", "w", encoding="utf-8") as f:
    f.write(resp.text)
print("✓ Downloaded pregen_tts_gpu.py")

# Download story files
os.makedirs("stories/telugu", exist_ok=True)
THEMES = ["Panchatantra", "Birbal", "Tenali Raman", "Festival", "Moral Story"]
for theme in THEMES:
    url = f"{REPO_BASE}/scripts/stories/telugu/{theme}.json"
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    path = f"stories/telugu/{theme}.json"
    with open(path, "w", encoding="utf-8") as f:
        f.write(resp.text)
    print(f"  ✓ {theme}.json")

print("\nAll files ready.")

## Step 6: Run TTS generation

This will:
1. Load `ai4bharat/indic-parler-tts` on GPU
2. For each story: chunk → generate WAV → merge → 128kbps MP3 → upload to Cloudinary
3. Save checkpoint after each story

**Estimated time:** ~1–2 minutes per story on T4 GPU.

In [ ]:
import subprocess, sys

args = [
    "--start", str(BATCH_START),
    "--count", str(BATCH_COUNT),
    "--lang", LANG,
    "--device", "cuda",
    "--stories-dir", "stories/telugu",
]

if CHECKPOINT_DIR:
    args += ["--checkpoint-dir", CHECKPOINT_DIR]

cmd = [sys.executable, "pregen_tts_gpu.py"] + args
print(f"Running: {' '.join(cmd)}")
print("=" * 70)

result = subprocess.run(cmd, check=True)
print(f"\nExit code: {result.returncode}")

## Done!

All generated audio is on Cloudinary. The web app's "Read" button will fetch it instantly.

**If interrupted:** Re-run the notebook — it skips already-uploaded stories via checkpoint + Cloudinary check.